# OceanEmbed — full ML pipeline (run top to bottom)

Surface satellite data (OSTIA SST, SMOS/SMAP SSS, DUACS SSH, OSCAR currents, CCMP winds) →
**ViT + FNO satellite-embedding encoder** → **U-Net decoder** → temperature at 35 depths (0–902 m),
trained on GLORYS with **physics-informed losses**, validated on independent **ARGO** floats.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU - switch runtime to T4 GPU"
# Drive is OPTIONAL (only used to back up results). If mounting fails, we continue without it.
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OK = True
except Exception as e:
    print("Drive not mounted (", e, ") -> continuing; results stay in /content, download them at the end.")


In [ ]:
%cd /content
!rm -rf Ocean_Embed
!git clone -q -b pinn-pipeline https://github.com/dhriti-goswami/Ocean_Embed.git
%cd /content/Ocean_Embed
!pip install -q netCDF4
!git log --oneline -3

## 2. Data + paths
Downloads the team cube from the shared Drive link. Results are written locally and backed up to your Drive after each stage.


In [ ]:
import os, shutil
# Data: team cube shared on Google Drive (link from Kaashvi)
DATA_FILE_ID = "1lV9vZpQxbK7ReGp0NDgAU17JRoHtFsU_"
DATA_DRIVE   = "/content/drive/MyDrive/oceanembed/complete_model_ready_dataset.nc"  # fallback if download fails
SAVE_DRIVE   = "/content/drive/MyDrive/oceanembed/pinn_pipeline"                    # results backup

DATA = "/content/cube.nc"
if not os.path.exists(DATA):
    !gdown -q --fuzzy "https://drive.google.com/file/d/{DATA_FILE_ID}/view" -O {DATA}
if not os.path.exists(DATA) or os.path.getsize(DATA) < 1e6:
    print("gdown failed (file not shared as 'Anyone with the link'?) -> using DATA_DRIVE path")
    if DRIVE_OK: shutil.copy(DATA_DRIVE, DATA)
    else: raise RuntimeError("Download failed and Drive not mounted: share the file as 'Anyone with the link' and re-run")

RUNS, RESULTS, OUTPUTS = "/content/runs", "/content/results", "/content/outputs"
for d in (RUNS, RESULTS, OUTPUTS): os.makedirs(d, exist_ok=True)

def backup():
    if not DRIVE_OK:
        print("(no Drive - skipping backup)"); return
    os.makedirs(SAVE_DRIVE, exist_ok=True)
    for d in (RUNS, RESULTS, OUTPUTS):
        shutil.copytree(d, os.path.join(SAVE_DRIVE, os.path.basename(d)), dirs_exist_ok=True)
    os.sync()
    print("backed up to", SAVE_DRIVE)

import xarray as xr
_ds = xr.open_dataset(DATA)
print("data:", round(os.path.getsize(DATA)/1e6, 1), "MB |", dict(_ds.sizes), "|", list(_ds.data_vars))
_ds.close()


## 3. Data sanity check

In [ ]:
from oceanembed.data import DataConfig, OceanData
data = OceanData(DataConfig(path=DATA))
print("inputs      :", data.channel_names)
print("grid        :", data.H, "x", data.W, " depths:", len(data.depths), f"({data.depths[0]:.1f}-{data.depths[-1]:.0f} m)")
print("salinity    :", data.has_salinity, "(enables density-based physics terms)")
print("ocean cells :", int(data.mask2d.sum()), "surface,", int(data.mask3d.sum()), "3D")
print("split (days):", {k: len(v) for k, v in data.idx.items()})
print("test days   :", str(data.times[data.idx['test'][0]])[:10], "->", str(data.times[data.idx['test'][-1]])[:10])
del data

## 4. Train three models
1. `unet` — plain U-Net (baseline, no ViT/FNO, no physics)
2. `oceanembed_nophys` — full ViT+FNO encoder + U-Net decoder, data loss only
3. `oceanembed_pinn` — same network + physics-informed losses (**main model**)

Each takes a few minutes on a T4.

In [ ]:
!python -m oceanembed.train --data {DATA} --out {RUNS}/unet --variant unet

In [ ]:
!python -m oceanembed.train --data {DATA} --out {RUNS}/oceanembed_nophys --variant oceanembed

In [ ]:
!python -m oceanembed.train --data {DATA} --out {RUNS}/oceanembed_pinn --variant oceanembed --physics
backup()

## 5. Evaluate on held-out days (vs GLORYS) + physics diagnostics

In [ ]:
RUN_LIST = f"{RUNS}/unet {RUNS}/oceanembed_nophys {RUNS}/oceanembed_pinn"
!python -m oceanembed.evaluate --data {DATA} --runs {RUN_LIST} --out {RESULTS}

## 6. Independent validation against ARGO floats (never used in training)

In [ ]:
!python -m oceanembed.argo --data {DATA} --runs {RUN_LIST} --out {RESULTS} --cache {RESULTS}/argo_cache.csv

## 7. Standardized output (daily, 0.25°, NetCDF) + latent embeddings

In [ ]:
!python -m oceanembed.infer --data {DATA} --ckpt {RUNS}/oceanembed_pinn/best.pt --out {OUTPUTS}
backup()

## 8. Figures

In [ ]:
from IPython.display import Image, display
from oceanembed import plots as P
figs = [
    P.per_depth_rmse(f"{RESULTS}/glorys_test_per_depth.csv", f"{RESULTS}/fig_rmse_by_depth.png"),
    P.training_curves([f"{RUNS}/unet", f"{RUNS}/oceanembed_nophys", f"{RUNS}/oceanembed_pinn"], f"{RESULTS}/fig_training.png"),
    P.argo_profiles(f"{RESULTS}/argo_matches_oceanembed_pinn.csv", f"{RESULTS}/fig_argo_profiles.png"),
    P.surface_vs_subsurface(f"{OUTPUTS}/oceanembed_temperature_daily_0p25.nc", 80, f"{RESULTS}/fig_depth_slices.png"),
]
for f in figs: display(Image(f))
backup()

## 9. Summary to paste back

In [ ]:
import pandas as pd, json
print(pd.read_csv(f"{RESULTS}/glorys_test_overall.csv").round(3).to_string(index=False))
print()
print(pd.read_csv(f"{RESULTS}/argo_summary.csv").round(3).to_string(index=False))
for r in ["unet","oceanembed_nophys","oceanembed_pinn"]:
    print(r, json.load(open(f"{RUNS}/{r}/summary.json")))

In [ ]:
# Download everything (results, figures, NetCDF outputs, checkpoints) as one zip
!cd /content && zip -qr oceanembed_results.zip results outputs runs
from google.colab import files
files.download("/content/oceanembed_results.zip")
